# 조선시대 신하 말투 Fine-tuning
**모델:** Qwen2.5-1.5B-Instruct  
**방법:** QLoRA (Unsloth)  
**환경:** Google Colab (무료 티어)

> ⚠️ 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 후 실행하세요.

## 1. 패키지 설치

In [ ]:
%%capture
!pip install unsloth
!pip install --upgrade pyarrow

## 2. 모델 로드
Unsloth로 4bit 양자화된 모델을 불러옵니다.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = 1024,
    load_in_4bit = True,
)
print("✅ 모델 로드 완료")

## 3. LoRA 설정
학습할 레이어만 선택해서 메모리를 아낍니다.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)
print("✅ LoRA 설정 완료")

## 4. 데이터 준비
`joseon_finetune_200.json`을 업로드한 뒤 실행하세요.

In [ ]:
# 파일 업로드
from google.colab import files
uploaded = files.upload()  # joseon_finetune_200.json 선택

In [ ]:
import json
from datasets import Dataset

with open("joseon_finetune_200.json", "r", encoding="utf-8") as f:
    raw = json.load(f)

# Qwen 채팅 템플릿 적용
def format_example(example):
    messages = [
        {"role": "user",      "content": example["instruction"]},
        {"role": "assistant", "content": example["output"]},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = Dataset.from_list(raw).map(format_example)
print(f"✅ 데이터셋 준비 완료: {len(dataset)}개")
print("\n--- 샘플 확인 ---")
print(dataset[0]["text"])

## 5. 학습 실행
약 5~10분 소요됩니다.

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = 1024,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        warmup_ratio = 0.1,
        lr_scheduler_type = "cosine",
        fp16 = True,
        logging_steps = 10,
        output_dir = "./joseon_output",
        report_to = "none",
    ),
)

trainer.train()
print("✅ 학습 완료")

## 6. 결과 비교
fine-tuning 전/후 말투를 비교합니다.

In [ ]:
def generate(prompt, model, max_new_tokens=200):
    FastLanguageModel.for_inference(model)
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        do_sample=True,
    )
    return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

# 테스트할 질문
test_questions = [
    "오늘 점심 뭐 먹을까?",
    "요즘 너무 피곤해.",
    "주말에 어디 가면 좋을까?",
]

for q in test_questions:
    print(f"Q: {q}")
    print(f"A: {generate(q, model)}")
    print("-" * 50)

## 7. 모델 저장 (선택)
LoRA 가중치만 저장합니다 (용량 작음).

In [ ]:
model.save_pretrained("joseon_lora")
tokenizer.save_pretrained("joseon_lora")

# 구글 드라이브에 저장하려면
# from google.colab import drive
# drive.mount('/content/drive')
# model.save_pretrained("/content/drive/MyDrive/joseon_lora")

print("✅ 저장 완료 → joseon_lora/")